In [1]:
#mounting to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#checking the files in the directory
import os
print(os.listdir('/content/drive/MyDrive/CamVid_data/'))

['CamVid.rar', 'CamVid']


In [3]:
#checking the files in the directory after extraction
import os
print(os.listdir('/content/drive/MyDrive/CamVid_data/CamVid/'))

['class_dict.csv', 'train_labels', 'val', 'val_labels', 'test', 'test_labels', 'train']


In [4]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/CamVid_data/CamVid/class_dict.csv')
print(df)

                 name    r    g    b
0              Animal   64  128   64
1             Archway  192    0  128
2           Bicyclist    0  128  192
3              Bridge    0  128   64
4            Building  128    0    0
5                 Car   64    0  128
6     CartLuggagePram   64    0  192
7               Child  192  128   64
8         Column_Pole  192  192  128
9               Fence   64   64  128
10       LaneMkgsDriv  128    0  192
11    LaneMkgsNonDriv  192    0   64
12          Misc_Text  128  128   64
13  MotorcycleScooter  192    0  192
14        OtherMoving  128   64   64
15       ParkingBlock   64  192  128
16         Pedestrian   64   64    0
17               Road  128   64  128
18       RoadShoulder  128  128  192
19           Sidewalk    0    0  192
20         SignSymbol  192  128  128
21                Sky  128  128  128
22     SUVPickupTruck   64  128  192
23        TrafficCone    0    0   64
24       TrafficLight    0   64   64
25              Train  192   64  128
2

In [5]:
#creating a dictionary to map the rgb values to class labels
df = pd.read_csv('/content/drive/MyDrive/CamVid_data/CamVid/class_dict.csv')
rgb_to_class={}
for index, rows in df.iterrows():
    rgb = (rows['r'], rows['g'], rows['b'])
    rgb_to_class[rgb] = index
print(rgb_to_class)
print(f"total number of classes: {len(rgb_to_class)}")


{(64, 128, 64): 0, (192, 0, 128): 1, (0, 128, 192): 2, (0, 128, 64): 3, (128, 0, 0): 4, (64, 0, 128): 5, (64, 0, 192): 6, (192, 128, 64): 7, (192, 192, 128): 8, (64, 64, 128): 9, (128, 0, 192): 10, (192, 0, 64): 11, (128, 128, 64): 12, (192, 0, 192): 13, (128, 64, 64): 14, (64, 192, 128): 15, (64, 64, 0): 16, (128, 64, 128): 17, (128, 128, 192): 18, (0, 0, 192): 19, (192, 128, 128): 20, (128, 128, 128): 21, (64, 128, 192): 22, (0, 0, 64): 23, (0, 64, 64): 24, (192, 64, 128): 25, (128, 128, 0): 26, (192, 128, 192): 27, (64, 0, 64): 28, (192, 192, 0): 29, (0, 0, 0): 30, (64, 192, 0): 31}
total number of classes: 32


In [6]:
import numpy as np 
from PIL import Image

def rgbmask_to_class(maskpath):
    mask = np.array(Image.open(maskpath).convert('RGB'))
    class_mask = np.full((mask.shape[0], mask.shape[1]),255,  dtype=np.uint8)
    for rgb, class_label in rgb_to_class.items():
        class_mask[np.all(mask == rgb, axis=-1)] = class_label
    return class_mask


In [7]:
test_mask = rgbmask_to_class('/content/drive/MyDrive/CamVid_data/CamVid/test_labels/0001TP_006690_L.png')
print(test_mask.shape)
print(np.unique(test_mask))

(720, 960)
[ 4  5  8 10 12 14 16 17 19 21 22 24 26 27 30]


In [8]:
# check a specific pixel manually
mask_rgb = np.array(Image.open('/content/drive/MyDrive/CamVid_data/CamVid/test_labels/0001TP_006690_L.png').convert('RGB'))

# pick a pixel, say (100, 100)
pixel = mask_rgb[100, 100]
print(f"RGB at (100,100): {tuple(pixel)}")
print(f"Mapped class: {rgb_to_class.get(tuple(pixel), 'NOT FOUND')}")

RGB at (100,100): (np.uint8(128), np.uint8(0), np.uint8(0))
Mapped class: 4


In [9]:
#loading the data 
import os
import torch
from PIL import Image 
from torch.utils.data import Dataset, DataLoader

class Camviddataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir 
        self.images = os.listdir(image_dir)
        self.mask_dir = mask_dir 
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir,self.images[idx].replace('.png', '_L.png'))
        image = np.array(Image.open(image_path).convert('RGB'))
        mask = rgbmask_to_class(mask_path)
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
        return image, mask.long()

In [10]:
#defining the transformations and using albumentations for data augmentation and normalization , instead of v2 
import albumentations as A
from albumentations.pytorch import ToTensorV2
transform =A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomCrop(height=512, width=512),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
]) 

In [11]:

base = '/content/drive/MyDrive/CamVid_data/CamVid'

sample_test = Camviddataset(
    image_dir=f"{base}/test",
    mask_dir=f"{base}/test_labels",
    transform=transform
)

print(len(sample_test))
image, mask = sample_test[0]
print(image.shape, mask.shape)
print(mask.unique())

232
torch.Size([3, 512, 512]) torch.Size([512, 512])
tensor([ 4,  5,  8, 12, 14, 16, 17, 19, 21, 22, 24, 26, 27, 30])


In [ ]:
#creating val_transform and test_transform 
import albumentations as A
from albumentations.pytorch import ToTensorV2
test_transform = A.Compose([
    A.Resize(height=512, width=512),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

In [13]:
#creating datasets
base = '/content/drive/MyDrive/CamVid_data/CamVid'
train_dataset=Camviddataset(image_dir=f"{base}/train",
                            mask_dir=f"{base}/train_labels",
                            transform=transform)
test_datasets=Camviddataset(image_dir=f"{base}/test",
                            mask_dir=f"{base}/test_labels",
                            transform=test_transform)
val_datasets=Camviddataset(image_dir=f"{base}/val",
                           mask_dir=f"{base}/val_labels",
                           transform=test_transform)

print(len(train_dataset), len(test_datasets), len(val_datasets))

369 232 100


In [14]:
#turning into dataloader
from torch.utils.data import dataloader

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0, drop_last=True)
test_loader = DataLoader(test_datasets, batch_size=2, shuffle=False, num_workers=0, drop_last=True)
val_loader = DataLoader(val_datasets, batch_size=2,shuffle=False, num_workers=0, drop_last=True)
print(len(train_loader), len(test_loader), len(val_loader))

184 116 50


In [15]:
#creating Unet from scratch 
import torch 
import torch.nn as nn
 
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )
    def forward(self, x):
       return self.conv(x)
    
    
class EncoderBlock(nn.Module):
    def __init__(self, in_channels:int, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
    def forward (self, x):
     skip = self.conv(x)
     pooled = self.pool(skip)
     return skip , pooled
    
#BOTTLENECK 
class Bottleneck(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)
    
    def forward(self, x):
        return self.conv(x)

#Decoder
class DecoderBlock(nn.Module):
    def __init__(self,  in_channels:int, out_channels:int):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x, skip):
        upsampled = self.upsample(x)
        concatenated = torch.cat([upsampled, skip], dim=1)
        return self.conv(concatenated)

#Assembling whole Unet class 
class UNet(nn.Module):
    def __init__(self, in_channels:int, num_classes:int):
        super().__init__()
        self.enc1 = EncoderBlock(in_channels, 64)
        self.enc2 = EncoderBlock(64, 128)
        self.enc3 = EncoderBlock(128, 256)
        self.enc4 = EncoderBlock(256, 512)

        self.BottleNeck = Bottleneck(512, 1024)

        self.Decoder1 = DecoderBlock(1024, 512)
        self.Decoder2 = DecoderBlock(512, 256 )
        self.Decoder3 = DecoderBlock(256, 128)
        self.Decoder4 = DecoderBlock(128, 64)

        self.output = nn.Conv2d(64, num_classes, kernel_size=1)
    def forward(self, x ):
        skip1, x = self.enc1(x)
        skip2, x = self.enc2(x)
        skip3, x = self.enc3(x)
        skip4, x = self.enc4(x)

        x = self.BottleNeck(x)

        x = self.Decoder1(x, skip4)
        x = self.Decoder2(x, skip3)
        x = self.Decoder3(x, skip2)
        x = self.Decoder4(x, skip1)
        return self.output(x)
    

In [16]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [17]:
num_classes= 32
model = UNet(in_channels=3, num_classes=32)
model = model.to(device)

In [18]:
!pip  install torchinfo
from torchinfo import summary
summary(model)


Layer (type:depth-idx)                   Param #
UNet                                     --
├─EncoderBlock: 1-1                      --
│    └─DoubleConv: 2-1                   --
│    │    └─Sequential: 3-1              38,976
│    └─MaxPool2d: 2-2                    --
├─EncoderBlock: 1-2                      --
│    └─DoubleConv: 2-3                   --
│    │    └─Sequential: 3-2              221,952
│    └─MaxPool2d: 2-4                    --
├─EncoderBlock: 1-3                      --
│    └─DoubleConv: 2-5                   --
│    │    └─Sequential: 3-3              886,272
│    └─MaxPool2d: 2-6                    --
├─EncoderBlock: 1-4                      --
│    └─DoubleConv: 2-7                   --
│    │    └─Sequential: 3-4              3,542,016
│    └─MaxPool2d: 2-8                    --
├─Bottleneck: 1-5                        --
│    └─DoubleConv: 2-9                   --
│    │    └─Sequential: 3-5              14,161,920
├─DecoderBlock: 1-6                      -

In [19]:
#optimizer , loss function and schedular
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-6 )
loss_fn = nn.CrossEntropyLoss(ignore_index=255, label_smoothing=0.1)

In [ ]:
#mIoU
def mIoU(preds, masks, num_classes, ignore_index=255):
    preds = torch.argmax(preds, dim=1)
    iou_list = []
    for cls in range(num_classes):
        valid = masks != ignore_index 
        preds_cls = (preds == cls) & valid 
        masks_cls = (masks == cls) & valid
        intersection = (preds_cls & masks_cls).sum().item()
        union = (preds_cls | masks_cls).sum().item()
        if  union == 0:
            continue

        iou_list.append( intersection / union)
    return sum(iou_list)  / len(iou_list) if iou_list else 0.0


In [21]:
#training step 
import torch 
import torch.nn as nn
def train_step(model:torch.nn.Module,
               dataloader:torch.utils.data.DataLoader,
               optimizer:torch.optim,
               loss_fn:torch.nn,
               device:torch.device):
    model.train()
    train_loss  = 0 
    for batch, (X, Y) in enumerate(dataloader):
        X, Y = X.to(device), Y.to(device)
        optimizer.zero_grad()
        y_pred = model(X)
        loss = loss_fn(y_pred , Y)
        train_loss += loss.item()
        loss.backward()
        optimizer.step()
    train_loss = train_loss / len(dataloader)
    print(f" Train loss:{train_loss:.4f} ")
     


In [22]:
#creating test step 
def test_step(model:torch.nn.Module,
              dataloader:torch.utils.data.DataLoader,
              loss_fn:torch.nn.Module,
              metric,
              device:torch.device
              ):
    model.eval()
    test_loss, test_acc = 0, 0 
    with torch.inference_mode():
        for batch, (X,Y) in enumerate(dataloader):
            X, Y = X.to(device), Y.to(device) 
            y_preds = model(X)
            loss = loss_fn(y_preds, Y)
            test_loss += loss.item()
            test_acc += metric(preds=y_preds, masks=Y, num_classes=32, ignore_index=255)
        test_loss = test_loss / len(dataloader)
        test_acc = test_acc / len(dataloader)
    print(f" Test loss :{test_loss:.4f} | Test accuracy:{test_acc:.4f}%")


In [23]:
#val step 
def val_step(model:torch.nn.Module,
             dataloader:torch.utils.data.DataLoader,
             loss_fn:torch.nn.Module,
             metric,
             device:torch.device):
    model.eval()
    val_loss, val_acc =0, 0 
    with torch.inference_mode():
        for batch, (X,Y) in enumerate(dataloader):
            X, Y = X.to(device), Y.to(device)
            y_preds = model(X)
            loss = loss_fn(y_preds, Y)
            val_loss += loss.item()
            val_acc += metric(preds=y_preds, masks=Y, num_classes=32, ignore_index=255)
        val_loss = val_loss / len(dataloader)
        val_acc = val_acc / len(dataloader)
        print(f" valulation loss:{val_loss:.4f} | accuarcy:{val_acc:.4f}%")

In [24]:
#calulating total time 
def total_train_time(start:float, end:float, device:torch.device):
    total_time = end - start 
    min = int( total_time // 60)
    sec = total_time%60
    print(f" Total time spend on device:{device} is {min}minutes and {sec:.2f}seconds")
    return total_time

In [25]:
#clearing cache
torch.cuda.empty_cache()
print(torch.cuda.memory_allocated() / 1024**2, "MB allocated")
print(torch.cuda.memory_reserved() / 1024**2, "MB reserved")

120.36376953125 MB allocated
136.0 MB reserved


In [26]:
from tqdm.auto import tqdm
from timeit import default_timer as timer

start_time = timer()
epochs = 1


for epoch in tqdm(range(epochs)):
    train_step(model=model,
               dataloader=train_loader,
               optimizer=optimizer,
               loss_fn=loss_fn,
               device=device)
    test_step(model=model,
               dataloader=test_loader,
               loss_fn=loss_fn,
               metric=mIoU,
               device=device)
    val_step(model=model,
               dataloader=val_loader,
               metric=mIoU,
               loss_fn=loss_fn,
               device=device)
    
end_time = timer()
training_time = total_train_time(start=start_time, end=end_time, device=device)
    

  0%|          | 0/1 [00:00<?, ?it/s]

 Train loss:2.3747 
 Test loss :2.1960 | Test accuracy:0.1127%
 valulation loss:1.9533 | accuarcy:0.1364%
 Total time spend on device:cuda is 18minutes and 36.88seconds
